In [44]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running on Google Colab. Setting up Kaggle data...")
    from google.colab import userdata
    import json

    # Setup Kaggle API
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

    !mkdir -p ~/.kaggle
    kaggle_config = {
        "username": os.environ["KAGGLE_USERNAME"],
        "key": os.environ["KAGGLE_KEY"],
    }
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(kaggle_config, f)
    !chmod 600 ~/.kaggle/kaggle.json

    # Download data if it doesn't exist yet
    if not os.path.exists("data"):
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded and unzipped.")
else:
    if not os.path.exists("data"):
        print("Download data ...")
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded")
    else:
        print("Data already exists")

Running on Google Colab. Setting up Kaggle data...


In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("Running on Google Colab. Setting up Kaggle data...")
    from google.colab import userdata
    import json

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !mkdir -p ~/.kaggle
    kaggle_config = {
        "username": os.environ["KAGGLE_USERNAME"],
        "key": os.environ["KAGGLE_KEY"],
    }
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(kaggle_config, f)
    !chmod 600 ~/.kaggle/kaggle.json

    if not os.path.exists("data"):
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded and unzipped.")
else:
    if not os.path.exists("data"):
        print("Download data ...")
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded")
    else:
        print("Data already exists")

import re
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# Feature engineering — this is what moves accuracy, not just tuning.
# Fit any *statistics* (median age by group, etc.) on TRAIN ONLY, then apply
# the same fitted statistics to both train and test to avoid leakage.
# ---------------------------------------------------------------------------

RARE_TITLE_MAP = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Rare", "Countess": "Rare", "Capt": "Rare", "Col": "Rare",
    "Don": "Rare", "Dr": "Rare", "Major": "Rare", "Rev": "Rare",
    "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare",
}


def extract_title(name: str) -> str:
    match = re.search(r",\s*([^.]+)\.", name)
    raw_title = match.group(1).strip() if match else "Unknown"
    # Some names have a leading article, e.g. "the Countess" -- take the last
    # whitespace-separated token so the lookup below works correctly.
    title = raw_title.split()[-1] if raw_title else "Unknown"
    return RARE_TITLE_MAP.get(title, title)


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Pure feature creation -- no fitting/learning of statistics happens here."""
    df = df.copy()

    df["Title"] = df["Name"].apply(extract_title)

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # First letter of cabin = deck; missing cabin -> "U" (unknown), itself informative
    df["Deck"] = df["Cabin"].apply(lambda c: c[0] if isinstance(c, str) and len(c) > 0 else "U")

    df["Embarked"] = df["Embarked"].fillna("S")  # mode of the real Titanic dataset

    return df


def fit_age_lookup(df: pd.DataFrame) -> pd.Series:
    """Median age per (Title, Pclass) group, computed on TRAIN only."""
    return df.groupby(["Title", "Pclass"])["Age"].median()


def apply_age_imputation(df: pd.DataFrame, age_lookup: pd.Series, global_median_age: float) -> pd.DataFrame:
    df = df.copy()
    missing = df["Age"].isna()
    if missing.any():
        filled = df.loc[missing].apply(
            lambda r: age_lookup.get((r["Title"], r["Pclass"]), global_median_age), axis=1
        )
        df.loc[missing, "Age"] = filled
    return df


CATEGORICAL_COLS = ["Sex", "Embarked", "Title", "Deck"]
NUMERIC_COLS = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]


def build_design_matrix(df: pd.DataFrame, fare_median: float) -> pd.DataFrame:
    df = df.copy()
    df["Fare"] = df["Fare"].fillna(fare_median)
    feats = df[NUMERIC_COLS + CATEGORICAL_COLS]
    feats = pd.get_dummies(feats, columns=CATEGORICAL_COLS, drop_first=True)
    return feats


# ---------------------------------------------------------------------------
# Load + engineer training data
# ---------------------------------------------------------------------------
df = pd.read_csv("data/train.csv")
df = engineer_features(df)

age_lookup = fit_age_lookup(df)
global_median_age = df["Age"].median()
df = apply_age_imputation(df, age_lookup, global_median_age)

fare_median = df["Fare"].median()

Y = df["Survived"]
X_df = build_design_matrix(df, fare_median)

scaler = MinMaxScaler()
X = scaler.fit_transform(X_df)
feature_columns = X_df.columns  # remember column order/identity for the test set

x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, shuffle=True, stratify=Y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ---------------------------------------------------------------------------
# Step 1: search across model families.
# ---------------------------------------------------------------------------
candidates = {
    "SVC": (
        SVC(probability=True),
        {
            "C": [0.5, 1, 5, 10, 50],
            "gamma": ["scale", "auto", 0.01, 0.05, 0.1],
            "kernel": ["rbf"],
        },
    ),
    "LogisticRegression": (
        LogisticRegression(max_iter=3000),
        {
            "C": [0.01, 0.1, 1, 10, 100],
            "solver": ["lbfgs"],
        },
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=RANDOM_STATE),
        {
            "n_estimators": [200, 400, 600],
            "max_depth": [4, 6, 8, None],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"],
        },
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.03, 0.05, 0.1],
            "max_depth": [2, 3, 4],
            "subsample": [0.8, 1.0],
        },
    ),
    "KNN": (
        KNeighborsClassifier(),
        {
            "n_neighbors": [5, 7, 9, 11, 15, 21],
            "weights": ["uniform", "distance"],
            "p": [1, 2],
        },
    ),
}

results = []
fitted_searches = {}

for name, (estimator, grid) in candidates.items():
    search = GridSearchCV(
        estimator=estimator,
        param_grid=grid,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(x_train, y_train)
    fitted_searches[name] = search
    results.append({
        "model": name,
        "best_cv_accuracy": search.best_score_,
        "best_params": search.best_params_,
    })
    print(f"{name:18s} best CV accuracy = {search.best_score_:.4f}  params = {search.best_params_}")

results_df = pd.DataFrame(results).sort_values("best_cv_accuracy", ascending=False).reset_index(drop=True)
print("\nModel ranking (by CV accuracy):")
print(results_df)

# ---------------------------------------------------------------------------
# Step 2: build a soft-voting ensemble of the top 3 tuned models.
# Ensembling different model families often beats any single model on a
# small, noisy dataset like Titanic because their errors are less correlated.
# ---------------------------------------------------------------------------
top3_names = results_df["model"].head(3).tolist()
voting_estimators = [(name, fitted_searches[name].best_estimator_) for name in top3_names]

ensemble = VotingClassifier(estimators=voting_estimators, voting="soft")
ensemble_cv_scores = []
for train_idx, val_idx in cv.split(x_train, y_train):
    ens_clone = clone(ensemble)
    ens_clone.fit(x_train[train_idx], y_train.iloc[train_idx])
    ensemble_cv_scores.append(
        accuracy_score(y_train.iloc[val_idx], ens_clone.predict(x_train[val_idx]))
    )
ensemble_cv_mean = float(np.mean(ensemble_cv_scores))
print(f"\n{'VotingEnsemble':18s} best CV accuracy = {ensemble_cv_mean:.4f}  (top3 = {top3_names})")

# ---------------------------------------------------------------------------
# Step 3: pick the overall winner — best single tuned model vs. the ensemble —
# by CV score, then sanity-check against the held-out split.
# ---------------------------------------------------------------------------
best_single_name = results_df.loc[0, "model"]
best_single_score = results_df.loc[0, "best_cv_accuracy"]

if ensemble_cv_mean > best_single_score:
    chosen_name = "VotingEnsemble"
    chosen_cv_score = ensemble_cv_mean
    chosen_model = ensemble  # not yet fit on full x_train; fit below
else:
    chosen_name = best_single_name
    chosen_cv_score = best_single_score
    chosen_model = fitted_searches[best_single_name].best_estimator_

chosen_model = clone(chosen_model)
chosen_model.fit(x_train, y_train)

p = chosen_model.predict(x_test)
holdout_acc = accuracy_score(y_test, p)

print(f"\nSelected model: {chosen_name}")
print(f"CV accuracy:      {chosen_cv_score:.4f}")
print(f"Holdout accuracy: {holdout_acc:.4f}")

gap = chosen_cv_score - holdout_acc
if gap > 0.07:
    print(f"WARNING: CV/holdout gap is {gap:.3f} — possible overfitting, "
          f"consider a simpler model or stronger regularization before submitting.")

# ---------------------------------------------------------------------------
# Step 4: refit the chosen model on ALL training data before predicting on
# the real Kaggle test set.
# ---------------------------------------------------------------------------
final_model = clone(chosen_model)
final_model.fit(X, Y)

# ---------------------------------------------------------------------------
# Load + engineer test data using statistics FIT ON TRAIN ONLY
# (age_lookup, global_median_age, fare_median, scaler, feature_columns)
# ---------------------------------------------------------------------------
df_test = pd.read_csv("data/test.csv")
df_test = engineer_features(df_test)
df_test = apply_age_imputation(df_test, age_lookup, global_median_age)

passenger_ids = df_test["PassengerId"]
X_test_df = build_design_matrix(df_test, fare_median)

# Align test columns to train columns (handles categories present in one
# split but not the other, e.g. a rare Deck/Title only seen in test).
X_test_df = X_test_df.reindex(columns=feature_columns, fill_value=0)

X_test_final = scaler.transform(X_test_df)

preds = final_model.predict(X_test_final)

# ---------------------------------------------------------------------------
# Build submission
# ---------------------------------------------------------------------------
output = pd.DataFrame({"PassengerId": passenger_ids, "Survived": preds})
output.to_csv("submission.csv", index=False)
print("\nsubmission.csv written:", output.shape)

if IN_COLAB:
    get_ipython().system('kaggle competitions submit -c titanic -f submission.csv -m "Message"')
else:
    os.system('kaggle competitions submit -c titanic -f submission.csv -m "Message"')

Running on Google Colab. Setting up Kaggle data...
SVC                best CV accuracy = 0.8259  params = {'C': 10, 'gamma': 0.05, 'kernel': 'rbf'}
LogisticRegression best CV accuracy = 0.8301  params = {'C': 10, 'solver': 'lbfgs'}
